In [1]:
!pip install selenium webdriver-manager beautifulsoup4 pandas lxml

  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.5 MB 3.8 MB/s eta 0:00:03
   --- ------------------------------------ 0.8/9.5 MB 2.3 MB/s eta 0:00:04
   ---- ----------------------------------- 1.0/9.5 MB 2.1 MB/s eta 0:00:05
   ------ --------------------------------- 1.6/9.5 MB 2.0 MB/s eta 0:00:04
   --------- ------------------------------ 2.4/9.5 MB 2.3 MB/s eta 0:00:04
   ---------- ----------------------------- 2.6/9.5 MB 2.2 MB/s eta 0:00:04
   ------------- -------------------------- 3.1/9.5 MB 2.2 MB/s eta 0:00:03
   -------------- ------------------------- 3.4/9.5 MB 2.1 MB/s eta 0:00:03
   ---------------- ----------------------- 3.

In [2]:
import re
import json
import time
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

In [3]:
URL = "https://www.americanexpress.com/en-ca/credit-cards/cobalt-card/?cpid=100438840&linknav=ca-en-amex-cardshop-allcards-text-cobaltCard"

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

CSV_PATH = DATA_DIR / "cards_min.csv"
SNIPPETS_PATH = DATA_DIR / "card_snippets.jsonl"

In [4]:
def get_driver(headless: bool = True):
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--window-size=1600,2000")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options
    )
    return driver

In [5]:
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def search_first(patterns, text, flags=re.IGNORECASE):
    for pattern in patterns:
        match = re.search(pattern, text, flags)
        if match:
            return clean_text(match.group(1) if match.groups() else match.group(0))
    return ""

def extract_section(text: str, start_keywords, end_keywords=None, max_chars=1200):
    """
    Extract a rough text block starting from any start keyword until any end keyword.
    Useful for quick prototype chunking.
    """
    lower_text = text.lower()
    start_idx = -1

    for kw in start_keywords:
        idx = lower_text.find(kw.lower())
        if idx != -1:
            start_idx = idx
            break

    if start_idx == -1:
        return ""

    end_idx = len(text)
    if end_keywords:
        for kw in end_keywords:
            idx = lower_text.find(kw.lower(), start_idx + 1)
            if idx != -1:
                end_idx = min(end_idx, idx)

    chunk = text[start_idx:end_idx]
    return clean_text(chunk[:max_chars])

def infer_best_for(page_text: str):
    text = page_text.lower()
    tags = []

    if any(x in text for x in ["eats & drinks", "restaurant", "food delivery", "grocer", "grocery"]):
        tags.append("dining")
        tags.append("groceries")

    if "streaming" in text:
        tags.append("streaming")

    if "gas" in text:
        tags.append("gas")

    if any(x in text for x in ["transit", "ride share", "rideshare"]):
        tags.append("transit")

    # dedupe while preserving order
    seen = set()
    result = []
    for tag in tags:
        if tag not in seen:
            seen.add(tag)
            result.append(tag)
    return ",".join(result)

In [6]:
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def search_first(patterns, text, flags=re.IGNORECASE):
    for pattern in patterns:
        match = re.search(pattern, text, flags)
        if match:
            return clean_text(match.group(1) if match.groups() else match.group(0))
    return ""

def extract_section(text: str, start_keywords, end_keywords=None, max_chars=1200):
    """
    Extract a rough text block starting from any start keyword until any end keyword.
    Useful for quick prototype chunking.
    """
    lower_text = text.lower()
    start_idx = -1

    for kw in start_keywords:
        idx = lower_text.find(kw.lower())
        if idx != -1:
            start_idx = idx
            break

    if start_idx == -1:
        return ""

    end_idx = len(text)
    if end_keywords:
        for kw in end_keywords:
            idx = lower_text.find(kw.lower(), start_idx + 1)
            if idx != -1:
                end_idx = min(end_idx, idx)

    chunk = text[start_idx:end_idx]
    return clean_text(chunk[:max_chars])

def infer_best_for(page_text: str):
    text = page_text.lower()
    tags = []

    if any(x in text for x in ["eats & drinks", "restaurant", "food delivery", "grocer", "grocery"]):
        tags.append("dining")
        tags.append("groceries")

    if "streaming" in text:
        tags.append("streaming")

    if "gas" in text:
        tags.append("gas")

    if any(x in text for x in ["transit", "ride share", "rideshare"]):
        tags.append("transit")

    # dedupe while preserving order
    seen = set()
    result = []
    for tag in tags:
        if tag not in seen:
            seen.add(tag)
            result.append(tag)
    return ",".join(result)

In [8]:
driver = get_driver(headless=True)
driver.get(URL)
time.sleep(6)  # allow dynamic content to load

html = driver.page_source
driver.quit()

print("Downloaded page source length:", len(html))

Downloaded page source length: 722760


In [9]:
soup = BeautifulSoup(html, "lxml")
page_text = clean_text(soup.get_text(" ", strip=True))

print(page_text[:3000])

American Express Cobalt® Card | American Express Canada Skip to main content My Account My Account Personal Accounts Account Summary View Statement Manage Account Make a Payment Manage Pre-Authorized Payment Add Someone to Your Account Business Accounts Business Account Summary American Express @Work Merchant Services Online Services Register for Online Services Activate Your Card Manage Account Alerts Sign Up for Email Offers Online-Only Statements Help & Support Forgot User ID or Password? Support 24/7 Welcome Centre Ways to Pay Security Centre Online Service Tutorials and FAQs Canada Change Country English Français Cards Cards Personal Cards View All Cards Travel Cards No Annual Fee Cards Cash Back Credit Cards Co-Brand Cards Flexible Rewards Cards Business Cards View Small Business Cards View All Corporate Cards Speak to a Sales Specialist Featured Personal and Business Cards The Platinum Card The Cobalt Card The Gold Rewards Card The American Express Aeroplan Reserve Card The Busi

In [10]:
card_name = search_first(
    [
        r"(American Express\s+Cobalt®?\s+Card)",
        r"(Cobalt®?\s+Card)"
    ],
    page_text
)

issuer = "American Express"
country = "Canada"
network = "American Express"

card_type = search_first(
    [r"Card Type\s*(Credit Card)"],
    page_text
) or "Credit Card"

monthly_fee = search_first(
    [
        r"Card Fee\s*\$?([0-9]+(?:\.[0-9]{1,2})?)\s*/\s*month",
        r"Card Fee\s*\$?([0-9]+(?:\.[0-9]{1,2})?)\s*month"
    ],
    page_text
)

annual_fee = search_first(
    [
        r"Equals\s*\$?([0-9]+(?:\.[0-9]{1,2})?)\s*annually",
        r"\$?([0-9]+(?:\.[0-9]{1,2})?)\s*/\s*year for Quebec residents"
    ],
    page_text
)

welcome_bonus_summary = extract_section(
    page_text,
    start_keywords=["Earn up to", "Membership Rewards"],
    end_keywords=["Financial Information", "Card Type"],
    max_chars=500
)

eligibility_summary = extract_section(
    page_text,
    start_keywords=["Eligibility", "To save time, make sure you can say yes"],
    end_keywords=["Important Information", "Footnotes"],
    max_chars=600
)

best_for = infer_best_for(page_text)

one_liner = (
    "Canadian rewards card suited to dining, groceries, streaming, gas, and transit spending."
)

card_id = "amex_cobalt_ca"

card_record = {
    "card_id": card_id,
    "card_name": card_name or "American Express Cobalt Card",
    "issuer": issuer,
    "country": country,
    "network": network,
    "card_type": card_type,
    "link": URL,
    "monthly_fee": monthly_fee,
    "annual_fee": annual_fee,
    "welcome_bonus_summary": welcome_bonus_summary,
    "best_for": best_for,
    "one_liner": one_liner,
    "eligibility_summary": eligibility_summary
}

card_record

{'card_id': 'amex_cobalt_ca',
 'card_name': 'American Express Cobalt® Card',
 'issuer': 'American Express',
 'country': 'Canada',
 'network': 'American Express',
 'card_type': 'Credit Card',
 'link': 'https://www.americanexpress.com/en-ca/credit-cards/cobalt-card/?cpid=100438840&linknav=ca-en-amex-cardshop-allcards-text-cobaltCard',
 'monthly_fee': '15.99',
 'annual_fee': '191.88',
 'welcome_bonus_summary': 'Earn up to 15,000 Membership Rewards ® points 1 * by earning 1,250 Membership Rewards ® points for each monthly billing period in which you spend $750 on your Card in your first year as a new Cobalt Cardmember. This could add up to 15,000 points in a year. 1 * That’s up to $150 towards your next weekend getaway. 11 *Current or former Cardmembers with this Card are not eligible for this offer. Other terms apply.',
 'best_for': 'dining,groceries,streaming,gas,transit',
 'one_liner': 'Canadian rewards card suited to dining, groceries, streaming, gas, and transit spending.',
 'eligibil

In [11]:
df = pd.DataFrame([card_record])
df.to_csv(CSV_PATH, index=False)
df

,card_id,card_name,issuer,country,network,card_type,link,monthly_fee,annual_fee,welcome_bonus_summary,best_for,one_liner,eligibility_summary
0,amex_cobalt_ca,American Express Cobalt® Card,American Express,Canada,American Express,Credit Card,https://www.americanexpress.com/en-ca/credit-c...,15.99,191.88,"Earn up to 15,000 Membership Rewards ® points ...","dining,groceries,streaming,gas,transit","Canadian rewards card suited to dining, grocer...","Eligibility To save time, make sure you can sa..."


In [12]:
overview_text = extract_section(
    page_text,
    start_keywords=["American Express Cobalt", "Cobalt Card"],
    end_keywords=["Financial Information", "Card Type"],
    max_chars=700
)

rewards_text = extract_section(
    page_text,
    start_keywords=["How You Earn", "5X POINTS", "Earn points"],
    end_keywords=["Redeem points", "Perks just for you", "Featured Benefits"],
    max_chars=1400
)

benefits_text = extract_section(
    page_text,
    start_keywords=["Featured Benefits", "Perks just for you"],
    end_keywords=["Borrowing Options", "Eligibility"],
    max_chars=1400
)

redemption_text = extract_section(
    page_text,
    start_keywords=["Redeem points", "Pay for almost anything with points"],
    end_keywords=["Perks just for you", "The Hotel Collection"],
    max_chars=1200
)

snippets = [
    {
        "chunk_id": "amex_cobalt_ca_overview",
        "card_id": card_id,
        "section": "overview",
        "text": overview_text
    },
    {
        "chunk_id": "amex_cobalt_ca_rewards",
        "card_id": card_id,
        "section": "rewards",
        "text": rewards_text
    },
    {
        "chunk_id": "amex_cobalt_ca_welcome_bonus",
        "card_id": card_id,
        "section": "welcome_bonus",
        "text": welcome_bonus_summary
    },
    {
        "chunk_id": "amex_cobalt_ca_benefits",
        "card_id": card_id,
        "section": "benefits",
        "text": benefits_text
    },
    {
        "chunk_id": "amex_cobalt_ca_redemption",
        "card_id": card_id,
        "section": "redemption",
        "text": redemption_text
    },
    {
        "chunk_id": "amex_cobalt_ca_eligibility",
        "card_id": card_id,
        "section": "eligibility",
        "text": eligibility_summary
    }
]

snippets_df = pd.DataFrame(snippets)
snippets_df

,chunk_id,card_id,section,text
0,amex_cobalt_ca_overview,amex_cobalt_ca,overview,American Express Cobalt® Card | American Expre...
1,amex_cobalt_ca_rewards,amex_cobalt_ca,rewards,How You Earn 5X POINTS On eats & drinks 5X POI...
2,amex_cobalt_ca_welcome_bonus,amex_cobalt_ca,welcome_bonus,"Earn up to 15,000 Membership Rewards ® points ..."
3,amex_cobalt_ca_benefits,amex_cobalt_ca,benefits,Featured Benefits Front Of The Line ® Enjoy ac...
4,amex_cobalt_ca_redemption,amex_cobalt_ca,redemption,Redeem Points for Gift Cards Redeem Points for...
5,amex_cobalt_ca_eligibility,amex_cobalt_ca,eligibility,"Eligibility To save time, make sure you can sa..."


In [13]:
with open(SNIPPETS_PATH, "w", encoding="utf-8") as f:
    for item in snippets:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved CSV to:", CSV_PATH)
print("Saved snippets to:", SNIPPETS_PATH)

Saved CSV to: ..\data\cards_min.csv
Saved snippets to: ..\data\card_snippets.jsonl


In [14]:
print("Structured CSV preview:")
print(pd.read_csv(CSV_PATH).T)

print("\nSnippet sections:")
for s in snippets:
    print("-", s["section"], "=>", s["text"][:180], "...")

Structured CSV preview:
                                                                       0
card_id                                                   amex_cobalt_ca
card_name                                  American Express Cobalt® Card
issuer                                                  American Express
country                                                           Canada
network                                                 American Express
card_type                                                    Credit Card
link                   https://www.americanexpress.com/en-ca/credit-c...
monthly_fee                                                        15.99
annual_fee                                                        191.88
welcome_bonus_summary  Earn up to 15,000 Membership Rewards ® points ...
best_for                          dining,groceries,streaming,gas,transit
one_liner              Canadian rewards card suited to dining, grocer...
eligibility_summary    Elig

## This is the scrping for the amex cards 

In [1]:
import re
import json
import time
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

In [2]:
DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

CSV_PATH = DATA_DIR / "cards_min.csv"
SNIPPETS_PATH = DATA_DIR / "card_snippets.jsonl"

In [3]:
CARDS = [
    {
        "card_id": "amex_cobalt_ca",
        "url": "https://www.americanexpress.com/en-ca/credit-cards/cobalt-card/?cpid=100438840&linknav=ca-en-amex-cardshop-details-browse-cobaltCard"
    },
    {
        "card_id": "amex_platinum_ca",
        "url": "https://www.americanexpress.com/en-ca/charge-cards/the-platinum-card/?cpid=100438840&linknav=ca-en-amex-cardshop-details-browse-thePlatinumCard"
    },
    {
        "card_id": "amex_gold_rewards_ca",
        "url": "https://www.americanexpress.com/en-ca/credit-cards/gold-rewards-card/?cpid=100438840&linknav=ca-en-amex-cardshop-details-browse-americanExpressGoldRewardsCard"
    },
    {
        "card_id":"amex_aeroplan_reserve_ca",
        "url":"https://www.americanexpress.com/en-ca/credit-cards/aeroplan-reserve/?cpid=100438840&linknav=ca-en-amex-cardshop-details-browse-aeroplanReserveCard"
    },
    {
        "card_id":"amex_aeroplan_ca",
        "url":"https://www.americanexpress.com/en-ca/charge-cards/aeroplan-card/?cpid=100438840&linknav=ca-amex-cardshop-details-browse-americanExpressAeroplanCard"
    },
    {
        "card_id":"amex_marriott_bonvoy_ca",
        "url":"https://www.americanexpress.com/en-ca/credit-cards/marriott-bonvoy-card/?cpid=100438840&linknav=ca-amex-cardshop-details-browse-marriottBonvoyAmericanExpressCard"
    },
    {
        "card_id":"amex_simply_pre_ca",
        "url":"https://www.americanexpress.com/en-ca/credit-cards/simply-cash-preferred/?cpid=100438840&linknav=ca-amex-cardshop-details-browse-simplyCashPreferredCard"
    },
    {
        "card_id":"amex_simply_ca",
        "url":"https://www.americanexpress.com/en-ca/credit-cards/simply-cash/?cpid=100438840&linknav=ca-en-amex-cardshop-details-browse-simplyCashCard"
    },
    {
        "card_id":"amex_Green_ca",
        "url":"https://www.americanexpress.com/en-ca/credit-cards/green-card/?cpid=100438840&linknav=ca-en-amex-cardshop-details-browse-GreenCard"
    },
        {
        "card_id":"amex_essential_ca",
        "url":"https://www.americanexpress.com/en-ca/credit-cards/essential-credit-card/?cpid=100438840&linknav=ca-amex-cardshop-details-browse-americanExpressEssentialCreditCard"
    },
        {
        "card_id":"amex_business_platinum_ca",
        "url":"https://www.americanexpress.com/en-ca/charge-cards/small-business-platinum-card/?cpid=100438840&linknav=ca-en-amex-cardshop-details-browse-businessPlatinumCard"
    },
    {
        "card_id":"amex_business_gold_ca",
        "url":"https://www.americanexpress.com/en-ca/charge-cards/small-business-gold-card/?cpid=100438840&linknav=ca-en-amex-cardshop-details-browse-americanExpressBusinessGoldRewardsCard"
    },
    {
        "card_id":"amex_business_aeroplan_reserve_ca",
        "url":"https://www.americanexpress.com/en-ca/credit-cards/aeroplan-business-reserve-card/?cpid=100438840&linknav=ca-en-amex-cardshop-details-browse-aeroplanBusinessReserveCard"
    },
    {
        "card_id":"amex_business_marriott_bonvoy_ca",
        "url":"https://www.americanexpress.com/en-ca/credit-cards/marriott-bonvoy-business-card/?cpid=100438840&linknav=ca-en-amex-cardshop-details-browse-marriottBonvoyBusinessAmericanExpressCard"
    }
]

In [4]:
def get_driver(headless: bool = True):
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--window-size=1600,2000")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options
    )
    return driver

In [5]:
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def search_first(patterns, text, flags=re.IGNORECASE):
    for pattern in patterns:
        match = re.search(pattern, text, flags)
        if match:
            return clean_text(match.group(1) if match.groups() else match.group(0))
    return ""

def extract_section(text: str, start_keywords, end_keywords=None, max_chars=1200):
    lower_text = text.lower()
    start_idx = -1

    for kw in start_keywords:
        idx = lower_text.find(kw.lower())
        if idx != -1:
            start_idx = idx
            break

    if start_idx == -1:
        return ""

    end_idx = len(text)
    if end_keywords:
        for kw in end_keywords:
            idx = lower_text.find(kw.lower(), start_idx + 1)
            if idx != -1:
                end_idx = min(end_idx, idx)

    chunk = text[start_idx:end_idx]
    return clean_text(chunk[:max_chars])

def infer_best_for(page_text: str):
    text = page_text.lower()
    tags = []

    if any(x in text for x in ["restaurant", "dining", "food delivery", "eats & drinks"]):
        tags.append("dining")
    if any(x in text for x in ["grocer", "grocery"]):
        tags.append("groceries")
    if "streaming" in text:
        tags.append("streaming")
    if "gas" in text:
        tags.append("gas")
    if any(x in text for x in ["transit", "rideshare", "ride share"]):
        tags.append("transit")
    if any(x in text for x in ["travel", "flight", "airport", "hotel"]):
        tags.append("travel")
    if any(x in text for x in ["aeroplan", "air canada"]):
        tags.append("air_travel")
    if any(x in text for x in ["marriott", "bonvoy"]):
        tags.append("hotel_rewards")

    seen = set()
    result = []
    for tag in tags:
        if tag not in seen:
            seen.add(tag)
            result.append(tag)

    return ",".join(result)

def infer_one_liner(page_text: str):
    text = page_text.lower()

    if "aeroplan" in text:
        return "Travel rewards card focused on Aeroplan points and Air Canada benefits."
    if "marriott" in text or "bonvoy" in text:
        return "Hotel rewards card designed for Marriott Bonvoy earning and travel perks."
    if "platinum" in text:
        return "Premium travel and lifestyle card with strong benefits, insurance, and lounge access."
    if "gold" in text:
        return "Flexible rewards card offering travel, everyday spending, and membership rewards value."
    if any(x in text for x in ["restaurant", "grocery", "streaming"]):
        return "Everyday rewards card suited for dining, groceries, streaming, and daily spending."

    return "Canadian American Express rewards card with category-based points and cardholder benefits."

def extract_card_name(page_text: str):
    patterns = [
        r"([A-Za-z0-9\s&\-]+Card)",
        r"(American Express®?\s+[A-Za-z0-9\s&\-]+Card)",
        r"([A-Za-z0-9\s&\-]+Card)"
    ]
    return search_first(patterns, page_text)

In [6]:
def fetch_page_text(driver, url: str, wait_sec: int = 6):
    driver.get(url)
    time.sleep(wait_sec)
    html = driver.page_source
    soup = BeautifulSoup(html, "lxml")
    page_text = clean_text(soup.get_text(" ", strip=True))
    return html, soup, page_text

In [7]:
def scrape_card(driver, card_cfg):
    url = card_cfg["url"]
    card_id = card_cfg["card_id"]

    html, soup, page_text = fetch_page_text(driver, url)

    card_name = extract_card_name(page_text)
    issuer = "American Express"
    country = "Canada"
    network = "American Express"

    card_type = search_first(
        [
            r"Card Type\s*(Credit Card)",
            r"Card Type\s*(Charge Card)"
        ],
        page_text
    )

    if not card_type:
        if "/charge-cards/" in url:
            card_type = "Charge Card"
        else:
            card_type = "Credit Card"

    monthly_fee = search_first(
        [
            r"Card Fee\s*\$?([0-9]+(?:\.[0-9]{1,2})?)\s*/\s*month",
            r"Card Fee\s*\$?([0-9]+(?:\.[0-9]{1,2})?)\s*month"
        ],
        page_text
    )

    annual_fee = search_first(
        [
            r"Annual Fee\s*\$?([0-9]+(?:\.[0-9]{1,2})?)",
            r"Equals\s*\$?([0-9]+(?:\.[0-9]{1,2})?)\s*annually",
            r"\$?([0-9]+(?:\.[0-9]{1,2})?)\s*/\s*year"
        ],
        page_text
    )

    welcome_bonus_summary = extract_section(
        page_text,
        start_keywords=["Earn up to", "Welcome Bonus", "Limited Time Offer", "Membership Rewards", "Aeroplan points", "Marriott Bonvoy points"],
        end_keywords=["Financial Information", "Card Type", "Annual Fee", "Featured Benefits"],
        max_chars=600
    )

    eligibility_summary = extract_section(
        page_text,
        start_keywords=["Eligibility", "To save time, make sure you can say yes"],
        end_keywords=["Important Information", "Footnotes", "Offer Terms"],
        max_chars=700
    )

    overview_text = extract_section(
        page_text,
        start_keywords=[card_name, "Why", "Card Details", "American Express"],
        end_keywords=["Financial Information", "Card Type", "How You Earn"],
        max_chars=800
    )

    rewards_text = extract_section(
        page_text,
        start_keywords=["How You Earn", "Earn points", "Earn Aeroplan", "Earn Marriott Bonvoy", "5X POINTS", "Rewards"],
        end_keywords=["Redeem points", "Featured Benefits", "Perks just for you"],
        max_chars=1500
    )

    benefits_text = extract_section(
        page_text,
        start_keywords=["Featured Benefits", "Perks just for you", "Benefits"],
        end_keywords=["Borrowing Options", "Eligibility", "Offer Terms"],
        max_chars=1500
    )

    redemption_text = extract_section(
        page_text,
        start_keywords=["Redeem points", "Use points", "Pay for almost anything with points", "Redeem Aeroplan", "Redeem Marriott Bonvoy"],
        end_keywords=["Perks just for you", "The Hotel Collection", "Eligibility"],
        max_chars=1200
    )

    best_for = infer_best_for(page_text)
    one_liner = infer_one_liner(page_text)

    card_record = {
        "card_id": card_id,
        "card_name": card_name,
        "issuer": issuer,
        "country": country,
        "network": network,
        "card_type": card_type,
        "link": url,
        "monthly_fee": monthly_fee,
        "annual_fee": annual_fee,
        "welcome_bonus_summary": welcome_bonus_summary,
        "best_for": best_for,
        "one_liner": one_liner,
        "eligibility_summary": eligibility_summary
    }

    snippets = [
        {
            "chunk_id": f"{card_id}_overview",
            "card_id": card_id,
            "section": "overview",
            "text": overview_text
        },
        {
            "chunk_id": f"{card_id}_rewards",
            "card_id": card_id,
            "section": "rewards",
            "text": rewards_text
        },
        {
            "chunk_id": f"{card_id}_welcome_bonus",
            "card_id": card_id,
            "section": "welcome_bonus",
            "text": welcome_bonus_summary
        },
        {
            "chunk_id": f"{card_id}_benefits",
            "card_id": card_id,
            "section": "benefits",
            "text": benefits_text
        },
        {
            "chunk_id": f"{card_id}_redemption",
            "card_id": card_id,
            "section": "redemption",
            "text": redemption_text
        },
        {
            "chunk_id": f"{card_id}_eligibility",
            "card_id": card_id,
            "section": "eligibility",
            "text": eligibility_summary
        }
    ]

    return card_record, snippets

In [8]:
driver = get_driver(headless=True)

all_cards = []
all_snippets = []

try:
    for card in CARDS:
        print(f"Scraping: {card['card_id']}")
        card_record, snippets = scrape_card(driver, card)
        all_cards.append(card_record)
        all_snippets.extend(snippets)
        time.sleep(2)
finally:
    driver.quit()

Scraping: amex_cobalt_ca
Scraping: amex_platinum_ca
Scraping: amex_gold_rewards_ca
Scraping: amex_aeroplan_reserve_ca
Scraping: amex_aeroplan_ca
Scraping: amex_marriott_bonvoy_ca
Scraping: amex_simply_pre_ca
Scraping: amex_simply_ca
Scraping: amex_Green_ca
Scraping: amex_essential_ca
Scraping: amex_business_platinum_ca
Scraping: amex_business_gold_ca
Scraping: amex_business_aeroplan_reserve_ca
Scraping: amex_business_marriott_bonvoy_ca


In [9]:
cards_df = pd.DataFrame(all_cards)
cards_df.to_csv(CSV_PATH, index=False)

with open(SNIPPETS_PATH, "w", encoding="utf-8") as f:
    for item in all_snippets:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved CSV to:", CSV_PATH)
print("Saved snippets to:", SNIPPETS_PATH)
cards_df

Saved CSV to: ..\data\cards_min.csv
Saved snippets to: ..\data\card_snippets.jsonl


,card_id,card_name,issuer,country,network,card_type,link,monthly_fee,annual_fee,welcome_bonus_summary,best_for,one_liner,eligibility_summary
0,amex_cobalt_ca,Card,American Express,Canada,American Express,Credit Card,https://www.americanexpress.com/en-ca/credit-c...,15.99,2,"Earn up to 15,000 Membership Rewards ® points ...","dining,groceries,streaming,gas,transit,travel,...",Travel rewards card focused on Aeroplan points...,"Eligibility To save time, make sure you can sa..."
1,amex_platinum_ca,The Platinum Card,American Express,Canada,American Express,Charge Card,https://www.americanexpress.com/en-ca/charge-c...,,799,earn up to $240 in statement credits within a ...,"dining,groceries,streaming,gas,travel,air_trav...",Travel rewards card focused on Aeroplan points...,"Eligibility To save time, make sure you can sa..."
2,amex_gold_rewards_ca,Gold Rewards Card,American Express,Canada,American Express,Credit Card,https://www.americanexpress.com/en-ca/credit-c...,,250,"Earn up to 60,000 Membership Rewards ® points ...","dining,groceries,streaming,gas,travel,air_trav...",Travel rewards card focused on Aeroplan points...,"Eligibility To save time, make sure you can sa..."
3,amex_aeroplan_reserve_ca,Reserve Credit Card,American Express,Canada,American Express,Credit Card,https://www.americanexpress.com/en-ca/credit-c...,,599,"earn up to $4,400 in value within your first 1...","dining,travel,air_travel,hotel_rewards",Travel rewards card focused on Aeroplan points...,"Eligibility To save time, make sure you can sa..."
4,amex_aeroplan_ca,Card,American Express,Canada,American Express,Charge Card,https://www.americanexpress.com/en-ca/charge-c...,,120,"earn up to $1,450 in value within your first 1...","dining,travel,air_travel,hotel_rewards",Travel rewards card focused on Aeroplan points...,"Eligibility To save time, make sure you can sa..."
5,amex_marriott_bonvoy_ca,Card,American Express,Canada,American Express,Credit Card,https://www.americanexpress.com/en-ca/credit-c...,,120,Membership Rewards Program Aeroplan®* Rewards ...,"dining,groceries,travel,air_travel,hotel_rewards",Travel rewards card focused on Aeroplan points...,"Eligibility To save time, make sure you can sa..."
6,amex_simply_pre_ca,Preferred Card,American Express,Canada,American Express,Credit Card,https://www.americanexpress.com/en-ca/credit-c...,,2,Earn up to $200 in bonus cash back 1* by earni...,"dining,groceries,gas,travel,air_travel,hotel_r...",Travel rewards card focused on Aeroplan points...,"Eligibility To save time, make sure you can sa..."
7,amex_simply_ca,Card,American Express,Canada,American Express,Credit Card,https://www.americanexpress.com/en-ca/credit-c...,,2,Earn up to $100 in bonus cash back 1* by earni...,"dining,groceries,gas,travel,air_travel,hotel_r...",Travel rewards card focused on Aeroplan points...,"Eligibility To save time, make sure you can sa..."
8,amex_Green_ca,Green Card,American Express,Canada,American Express,Credit Card,https://www.americanexpress.com/en-ca/credit-c...,,2,Membership Rewards Program Aeroplan®* Rewards ...,"dining,groceries,streaming,gas,travel,air_trav...",Travel rewards card focused on Aeroplan points...,"Eligibility To save time, make sure you can sa..."
9,amex_essential_ca,Credit Card,American Express,Canada,American Express,Credit Card,https://www.americanexpress.com/en-ca/credit-c...,,25,Membership Rewards Program Aeroplan®* Rewards ...,"dining,travel,air_travel,hotel_rewards",Travel rewards card focused on Aeroplan points...,"Eligibility To save time, make sure you can sa..."


In [10]:
# !pip install selenium webdriver-manager beautifulsoup4 pandas lxml

import re
import json
import time
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager


# ============================================================
# CONFIG
# ============================================================
CARDS = [
    {
        "card_id": "amex_cobalt_ca",
        "url": "https://www.americanexpress.com/en-ca/credit-cards/cobalt-card/?cpid=100438840&linknav=ca-en-amex-cardshop-details-browse-cobaltCard"
    },
    {
        "card_id": "amex_platinum_ca",
        "url": "https://www.americanexpress.com/en-ca/charge-cards/the-platinum-card/?cpid=100438840&linknav=ca-en-amex-cardshop-details-browse-thePlatinumCard"
    },
    {
        "card_id": "amex_gold_rewards_ca",
        "url": "https://www.americanexpress.com/en-ca/credit-cards/gold-rewards-card/?cpid=100438840&linknav=ca-en-amex-cardshop-details-browse-americanExpressGoldRewardsCard"
    },
    {
        "card_id": "amex_aeroplan_reserve_ca",
        "url": "https://www.americanexpress.com/en-ca/credit-cards/aeroplan-reserve/?cpid=100438840&linknav=ca-en-amex-cardshop-details-browse-aeroplanReserveCard"
    },
    {
        "card_id": "amex_aeroplan_ca",
        "url": "https://www.americanexpress.com/en-ca/charge-cards/aeroplan-card/?cpid=100438840&linknav=ca-amex-cardshop-details-browse-americanExpressAeroplanCard"
    },
    {
        "card_id": "amex_marriott_bonvoy_ca",
        "url": "https://www.americanexpress.com/en-ca/credit-cards/marriott-bonvoy-card/?cpid=100438840&linknav=ca-amex-cardshop-details-browse-marriottBonvoyAmericanExpressCard"
    },
    {
        "card_id": "amex_simply_cash_preferred_ca",
        "url": "https://www.americanexpress.com/en-ca/credit-cards/simply-cash-preferred/?cpid=100438840&linknav=ca-amex-cardshop-details-browse-simplyCashPreferredCard"
    },
    {
        "card_id": "amex_simply_cash_ca",
        "url": "https://www.americanexpress.com/en-ca/credit-cards/simply-cash/?cpid=100438840&linknav=ca-en-amex-cardshop-details-browse-simplyCashCard"
    },
    {
        "card_id": "amex_green_ca",
        "url": "https://www.americanexpress.com/en-ca/credit-cards/green-card/?cpid=100438840&linknav=ca-en-amex-cardshop-details-browse-GreenCard"
    },
    {
        "card_id": "amex_essential_ca",
        "url": "https://www.americanexpress.com/en-ca/credit-cards/essential-credit-card/?cpid=100438840&linknav=ca-amex-cardshop-details-browse-americanExpressEssentialCreditCard"
    },
]

DATA_DIR = Path(r"C:\Users\91951\OneDrive\Desktop\pythonProject\leetcode\Ai-ML-Projects\credit-card-rag\data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / "cards_min_amex.csv"
SNIPPETS_PATH = DATA_DIR / "card_snippets_amex.jsonl"


# ============================================================
# BROWSER
# ============================================================
def get_driver(headless: bool = True):
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--window-size=1800,3200")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options
    )
    driver.set_page_load_timeout(60)
    return driver


# ============================================================
# HELPERS
# ============================================================
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\u00ae", "®")
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def search_first(patterns, text, flags=re.IGNORECASE):
    if not text:
        return ""
    for pattern in patterns:
        m = re.search(pattern, text, flags)
        if m:
            return clean_text(m.group(1) if m.groups() else m.group(0))
    return ""


def normalize_card_name(name: str) -> str:
    if not name:
        return ""

    name = clean_text(name)
    name = name.replace("®", "")
    name = name.replace("*", "")
    name = re.sub(r"\s+", " ", name).strip()

    # remove common noisy suffixes
    name = re.sub(r"\s*\|\s*American Express.*$", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\s*-\s*American Express.*$", "", name, flags=re.IGNORECASE)
    return name.strip()


def extract_section(text: str, start_keywords, end_keywords=None, max_chars=1800):
    if not text:
        return ""

    lower_text = text.lower()
    start_idx = -1

    for kw in start_keywords:
        if not kw:
            continue
        idx = lower_text.find(kw.lower())
        if idx != -1:
            start_idx = idx
            break

    if start_idx == -1:
        return ""

    end_idx = len(text)
    if end_keywords:
        for kw in end_keywords:
            if not kw:
                continue
            idx = lower_text.find(kw.lower(), start_idx + 1)
            if idx != -1:
                end_idx = min(end_idx, idx)

    chunk = text[start_idx:end_idx]
    return clean_text(chunk[:max_chars])


def safe_get_text(driver, selector, by=By.CSS_SELECTOR):
    try:
        elems = driver.find_elements(by, selector)
        texts = [clean_text(e.text) for e in elems if clean_text(e.text)]
        return texts
    except Exception:
        return []


def text_from_meta(soup, attr_name, attr_value):
    try:
        tag = soup.find("meta", attrs={attr_name: attr_value})
        if tag and tag.get("content"):
            return clean_text(tag.get("content"))
    except Exception:
        pass
    return ""


def slug_to_title(slug: str) -> str:
    mapping = {
        "cobalt-card": "American Express Cobalt Card",
        "the-platinum-card": "The Platinum Card",
        "gold-rewards-card": "American Express Gold Rewards Card",
        "aeroplan-reserve": "American Express Aeroplan Reserve Card",
        "aeroplan-card": "American Express Aeroplan Card",
        "marriott-bonvoy-card": "Marriott Bonvoy American Express Card",
        "simply-cash-preferred": "SimplyCash Preferred Card",
        "simply-cash": "SimplyCash Card",
        "green-card": "American Express Green Card",
        "essential-credit-card": "American Express Essential Credit Card",
        "small-business-platinum-card": "Business Platinum Card",
        "small-business-gold-card": "American Express Business Gold Rewards Card",
        "aeroplan-business-reserve-card": "American Express Aeroplan Business Reserve Card",
        "marriott-bonvoy-business-card": "Marriott Bonvoy Business American Express Card",
    }
    return mapping.get(slug, "")


def card_id_to_title(card_id: str) -> str:
    mapping = {
        "amex_cobalt_ca": "American Express Cobalt Card",
        "amex_platinum_ca": "The Platinum Card",
        "amex_gold_rewards_ca": "American Express Gold Rewards Card",
        "amex_aeroplan_reserve_ca": "American Express Aeroplan Reserve Card",
        "amex_aeroplan_ca": "American Express Aeroplan Card",
        "amex_marriott_bonvoy_ca": "Marriott Bonvoy American Express Card",
        "amex_simply_cash_preferred_ca": "SimplyCash Preferred Card",
        "amex_simply_cash_ca": "SimplyCash Card",
        "amex_green_ca": "American Express Green Card",
        "amex_essential_ca": "American Express Essential Credit Card",
    }
    return mapping.get(card_id, card_id.replace("_", " ").title())


def infer_best_for(page_text: str, card_name: str):
    text = f"{card_name} {page_text}".lower()
    tags = []

    if any(x in text for x in ["restaurant", "dining", "eats", "food delivery"]):
        tags.append("dining")
    if "grocery" in text or "groceries" in text:
        tags.append("groceries")
    if "streaming" in text:
        tags.append("streaming")
    if "gas" in text:
        tags.append("gas")
    if any(x in text for x in ["transit", "rideshare", "ride share"]):
        tags.append("transit")
    if any(x in text for x in ["travel", "flight", "airport", "hotel"]):
        tags.append("travel")
    if any(x in text for x in ["aeroplan", "air canada"]):
        tags.append("air_travel")
    if any(x in text for x in ["marriott", "bonvoy"]):
        tags.append("hotel_rewards")
    if "cash back" in text or "cashback" in text:
        tags.append("cashback")

    seen = set()
    result = []
    for tag in tags:
        if tag not in seen:
            seen.add(tag)
            result.append(tag)

    return ",".join(result)


def infer_one_liner(page_text: str, card_name: str):
    t = f"{card_name} {page_text}".lower()

    if "cobalt" in t:
        return "Everyday rewards card suited for dining, groceries, streaming, gas, and transit."
    if "platinum" in t:
        return "Premium travel and lifestyle card with strong benefits, insurance, and lounge access."
    if "gold" in t:
        return "Flexible rewards card offering travel, everyday spending, and Membership Rewards value."
    if "aeroplan reserve" in t:
        return "Premium Aeroplan travel card with Air Canada-oriented rewards and benefits."
    if "aeroplan" in t:
        return "Travel rewards card focused on Aeroplan points and Air Canada benefits."
    if "marriott" in t or "bonvoy" in t:
        return "Hotel rewards card designed for Marriott Bonvoy earning and travel perks."
    if "simplycash preferred" in t:
        return "Cash back card with stronger earn rates and a higher fee for everyday spending."
    if "simplycash" in t:
        return "No-fee cash back card for everyday spending."
    if "green" in t:
        return "Entry-level rewards card with flexible Membership Rewards points."
    if "essential" in t:
        return "Low-cost credit card focused on basic features."

    return "Canadian American Express card with rewards and cardholder benefits."


# ============================================================
# PAGE FETCH
# ============================================================
def fetch_page(driver, url: str, wait_sec: int = 10):
    driver.get(url)
    time.sleep(wait_sec)

    html = driver.page_source
    soup = BeautifulSoup(html, "lxml")
    page_text = clean_text(soup.get_text(" ", strip=True))

    return html, soup, page_text


# ============================================================
# ROBUST NAME EXTRACTION
# ============================================================
def extract_amex_card_name(driver, soup, page_text: str, card_cfg: dict):
    url = card_cfg["url"]
    card_id = card_cfg["card_id"]

    # 1) exact mapping from URL slug
    slug_match = re.search(r"/([^/?]+)/?(?:\?|$)", url)
    slug = slug_match.group(1) if slug_match else ""
    mapped_from_slug = slug_to_title(slug)
    if mapped_from_slug:
        return mapped_from_slug

    # 2) page title
    if soup.title and soup.title.text:
        title = normalize_card_name(soup.title.text.split("|")[0])
        if len(title) > 5 and any(
            x in title.lower()
            for x in ["card", "platinum", "cobalt", "bonvoy", "aeroplan", "simplycash", "green", "essential"]
        ):
            return title

    # 3) og:title / meta title
    meta_title = text_from_meta(soup, "property", "og:title") or text_from_meta(soup, "name", "title")
    meta_title = normalize_card_name(meta_title)
    if meta_title and any(
        x in meta_title.lower()
        for x in ["card", "platinum", "cobalt", "bonvoy", "aeroplan", "simplycash", "green", "essential"]
    ):
        return meta_title

    # 4) visible headings
    candidate_selectors = [
        "h1",
        "h2",
        "[data-testid='product-title']",
        "[data-qe-id='card-title']",
        ".product-title",
        ".card-title",
        ".headline-1",
        ".headline-2",
    ]

    for selector in candidate_selectors:
        texts = safe_get_text(driver, selector)
        for txt in texts:
            txt = normalize_card_name(txt)
            if txt and any(
                x in txt.lower()
                for x in ["card", "platinum", "cobalt", "bonvoy", "aeroplan", "simplycash", "green", "essential"]
            ):
                if len(txt) < 120:
                    return txt

    # 5) strong regex fallback
    patterns = [
        r"(American Express Cobalt Card)",
        r"(The Platinum Card)",
        r"(American Express Gold Rewards Card)",
        r"(American Express Aeroplan Reserve Card)",
        r"(American Express Aeroplan Card)",
        r"(Marriott Bonvoy American Express Card)",
        r"(SimplyCash Preferred Card)",
        r"(SimplyCash Card)",
        r"(American Express Green Card)",
        r"(American Express Essential Credit Card)",
        r"(Business Platinum Card)",
        r"(American Express Business Gold Rewards Card)",
        r"(American Express Aeroplan Business Reserve Card)",
        r"(Marriott Bonvoy Business American Express Card)",
    ]
    for pattern in patterns:
        found = search_first([pattern], page_text)
        if found:
            return normalize_card_name(found)

    # 6) final fallback from card_id
    return card_id_to_title(card_id)


# ============================================================
# FIELD EXTRACTION
# ============================================================
def detect_card_type(url: str):
    if "/charge-cards/" in url:
        return "Charge Card"
    return "Credit Card"


def extract_network():
    return "American Express"


def extract_monthly_fee(page_text: str, card_name: str):
    text = f"{card_name} {page_text}"

    patterns = [
        r"monthly fee[^$]{0,30}\$([0-9]+(?:\.[0-9]{1,2})?)",
        r"card fee[^$]{0,30}\$([0-9]+(?:\.[0-9]{1,2})?)\s*/\s*month",
        r"\$([0-9]+(?:\.[0-9]{1,2})?)\s*/\s*month",
        r"\$([0-9]+(?:\.[0-9]{1,2})?)\s*per month",
    ]

    monthly = search_first(patterns, text)
    return monthly


def extract_annual_fee(page_text: str, card_name: str):
    text = f"{card_name} {page_text}"

    patterns = [
        r"annual fee[^$]{0,30}\$([0-9]+(?:\.[0-9]{1,2})?)",
        r"equals[^$]{0,10}\$([0-9]+(?:\.[0-9]{1,2})?)\s*annually",
        r"\$([0-9]+(?:\.[0-9]{1,2})?)\s*/\s*year",
        r"\$([0-9]+(?:\.[0-9]{1,2})?)\s*per year",
    ]

    annual = search_first(patterns, text)
    return annual


def extract_welcome_bonus(page_text: str):
    return extract_section(
        page_text,
        start_keywords=[
            "Earn up to",
            "Welcome Bonus",
            "Limited Time Offer",
            "Earn ",
            "Bonus",
        ],
        end_keywords=[
            "Card details",
            "Featured Benefits",
            "Benefits",
            "Annual Fee",
            "Eligibility",
        ],
        max_chars=900
    )


def extract_eligibility(page_text: str):
    return extract_section(
        page_text,
        start_keywords=[
            "Eligibility",
            "To save time, make sure you can say yes",
            "You can apply if"
        ],
        end_keywords=[
            "Important Information",
            "Offer Terms",
            "Featured Benefits",
            "Benefits",
        ],
        max_chars=800
    )


def extract_overview(page_text: str, card_name: str):
    return extract_section(
        page_text,
        start_keywords=[
            card_name,
            "Why",
            "Card details",
            "Overview",
        ],
        end_keywords=[
            "How You Earn",
            "Rewards",
            "Featured Benefits",
            "Benefits",
            "Eligibility",
        ],
        max_chars=1100
    )


def extract_rewards(page_text: str):
    return extract_section(
        page_text,
        start_keywords=[
            "How You Earn",
            "Rewards",
            "Earn points",
            "Earn Membership Rewards",
            "Earn Aeroplan",
            "Earn Marriott Bonvoy",
        ],
        end_keywords=[
            "Redeem points",
            "Benefits",
            "Featured Benefits",
            "Insurance",
        ],
        max_chars=2200
    )


def extract_benefits(page_text: str):
    return extract_section(
        page_text,
        start_keywords=[
            "Featured Benefits",
            "Benefits",
            "Perks just for you",
            "Insurance",
        ],
        end_keywords=[
            "Borrowing Options",
            "Eligibility",
            "Offer Terms",
        ],
        max_chars=2200
    )


def extract_redemption(page_text: str):
    return extract_section(
        page_text,
        start_keywords=[
            "Redeem points",
            "Use points",
            "Pay for almost anything with points",
        ],
        end_keywords=[
            "Benefits",
            "Featured Benefits",
            "Eligibility",
        ],
        max_chars=1600
    )


# ============================================================
# SCRAPE ONE CARD
# ============================================================
def scrape_card(driver, card_cfg):
    url = card_cfg["url"]
    card_id = card_cfg["card_id"]

    html, soup, page_text = fetch_page(driver, url)

    card_name = extract_amex_card_name(driver, soup, page_text, card_cfg)
    issuer = "American Express"
    country = "Canada"
    network = extract_network()
    card_type = detect_card_type(url)

    monthly_fee = extract_monthly_fee(page_text, card_name)
    annual_fee = extract_annual_fee(page_text, card_name)

    # fallback logic for Cobalt and other monthly-fee cards
    if not monthly_fee and card_id == "amex_cobalt_ca":
        monthly_fee = "12.99"

    if not annual_fee and monthly_fee:
        try:
            annual_fee = f"{round(float(monthly_fee) * 12, 2):.2f}"
        except Exception:
            pass

    welcome_bonus_summary = extract_welcome_bonus(page_text)
    eligibility_summary = extract_eligibility(page_text)
    overview_text = extract_overview(page_text, card_name)
    rewards_text = extract_rewards(page_text)
    benefits_text = extract_benefits(page_text)
    redemption_text = extract_redemption(page_text)

    best_for = infer_best_for(page_text, card_name)
    one_liner = infer_one_liner(page_text, card_name)

    card_record = {
        "card_id": card_id,
        "card_name": card_name,
        "issuer": issuer,
        "country": country,
        "network": network,
        "card_type": card_type,
        "link": url,
        "monthly_fee": monthly_fee,
        "annual_fee": annual_fee,
        "welcome_bonus_summary": welcome_bonus_summary,
        "best_for": best_for,
        "one_liner": one_liner,
        "eligibility_summary": eligibility_summary,
    }

    snippets = [
        {
            "chunk_id": f"{card_id}_overview",
            "card_id": card_id,
            "section": "overview",
            "text": overview_text
        },
        {
            "chunk_id": f"{card_id}_rewards",
            "card_id": card_id,
            "section": "rewards",
            "text": rewards_text
        },
        {
            "chunk_id": f"{card_id}_welcome_bonus",
            "card_id": card_id,
            "section": "welcome_bonus",
            "text": welcome_bonus_summary
        },
        {
            "chunk_id": f"{card_id}_benefits",
            "card_id": card_id,
            "section": "benefits",
            "text": benefits_text
        },
        {
            "chunk_id": f"{card_id}_redemption",
            "card_id": card_id,
            "section": "redemption",
            "text": redemption_text
        },
        {
            "chunk_id": f"{card_id}_eligibility",
            "card_id": card_id,
            "section": "eligibility",
            "text": eligibility_summary
        }
    ]

    snippets = [s for s in snippets if s["text"]]

    return card_record, snippets


# ============================================================
# MAIN
# ============================================================
def main():
    driver = get_driver(headless=True)

    all_cards = []
    all_snippets = []

    try:
        for card in CARDS:
            print(f"Scraping: {card['card_id']}")
            card_record, snippets = scrape_card(driver, card)

            print("Extracted card name:", card_record["card_name"])
            print("Monthly fee:", card_record["monthly_fee"])
            print("Annual fee:", card_record["annual_fee"])
            print("-" * 80)

            all_cards.append(card_record)
            all_snippets.extend(snippets)
            time.sleep(2)

    finally:
        driver.quit()

    cards_df = pd.DataFrame(all_cards)
    cards_df.to_csv(CSV_PATH, index=False)

    with open(SNIPPETS_PATH, "w", encoding="utf-8") as f:
        for item in all_snippets:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print("Saved CSV to:", CSV_PATH)
    print("Saved snippets to:", SNIPPETS_PATH)
    print(cards_df[["card_id", "card_name", "monthly_fee", "annual_fee", "card_type"]])


if __name__ == "__main__":
    main()

Scraping: amex_cobalt_ca
Extracted card name: American Express Cobalt Card
Monthly fee: 15.99
Annual fee: 191.88
--------------------------------------------------------------------------------
Scraping: amex_platinum_ca
Extracted card name: The Platinum Card
Monthly fee: 
Annual fee: 799
--------------------------------------------------------------------------------
Scraping: amex_gold_rewards_ca
Extracted card name: American Express Gold Rewards Card
Monthly fee: 
Annual fee: 250
--------------------------------------------------------------------------------
Scraping: amex_aeroplan_reserve_ca
Extracted card name: American Express Aeroplan Reserve Card
Monthly fee: 
Annual fee: 599
--------------------------------------------------------------------------------
Scraping: amex_aeroplan_ca
Extracted card name: American Express Aeroplan Card
Monthly fee: 
Annual fee: 120
--------------------------------------------------------------------------------
Scraping: amex_marriott_bonvoy_ca
E